In [8]:
from pybaseball import bwar_bat, bwar_pitch, cache

cache.enable()

bat = bwar_bat()
print(bat.shape)
print(bat.columns.tolist())
bat[bat["year_ID"] == 2023].head()

(126547, 17)
['name_common', 'mlb_ID', 'player_ID', 'year_ID', 'team_ID', 'stint_ID', 'lg_ID', 'pitcher', 'G', 'PA', 'salary', 'runs_above_avg', 'runs_above_avg_off', 'runs_above_avg_def', 'WAR_rep', 'WAA', 'WAR']


,name_common,mlb_ID,player_ID,year_ID,team_ID,stint_ID,lg_ID,pitcher,G,PA,salary,runs_above_avg,runs_above_avg_off,runs_above_avg_def,WAR_rep,WAA,WAR
66,Fernando Abad,472551.0,abadfe01,2023,COL,1,NL,Y,0,0,100000.0,0.0,0.0,0.0,NaN,NaN,NaN
90,Andrew Abbott,671096.0,abbotan01,2023,CIN,1,NL,Y,0,0,NaN,0.0,0.0,0.0,NaN,NaN,NaN
95,Cory Abbott,676265.0,abbotco01,2023,WSN,1,NL,Y,0,0,NaN,0.0,0.0,0.0,NaN,NaN,NaN
226,CJ Abrams,682928.0,abramcj01,2023,WSN,1,NL,N,151,614,724200.0,14.8,10.8,12.8,1.98,1.47,3.45
237,Albert Abreu,656061.0,abreual01,2023,NYY,1,AL,Y,0,0,738700.0,0.0,0.0,0.0,NaN,NaN,NaN


In [9]:
import pandas as pd

In [3]:
print(bat["year_ID"].min(), bat["year_ID"].max())

pit = bwar_pitch()
print(pit.shape)
print(pit.columns.tolist())
pit[pit["year_ID"] == 2023].head()

1871 2026
(57884, 19)
['name_common', 'mlb_ID', 'player_ID', 'year_ID', 'team_ID', 'stint_ID', 'lg_ID', 'G', 'GS', 'RA', 'xRA', 'BIP', 'BIP_perc', 'salary', 'ERA_plus', 'WAR_rep', 'WAA', 'WAA_adj', 'WAR']


,name_common,mlb_ID,player_ID,year_ID,team_ID,stint_ID,lg_ID,G,GS,RA,xRA,BIP,BIP_perc,salary,ERA_plus,WAR_rep,WAA,WAA_adj,WAR
53911,Albert Abreu,656061.0,abreual01,2023,NYY,1,AL,45,0,39,32.313,165,0.0408,738700.0,91.967742,0.5697,-0.8145,0.0027,-0.24
53912,Bryan Abreu,650556.0,abreubr01,2023,HOU,1,AL,72,0,17,37.368,153,0.0379,745200.0,243.357143,0.6945,1.7064,0.1199,2.52
53913,Domingo Acevedo,642758.0,acevedo01,2023,OAK,1,AL,9,0,11,4.798,37,0.0086,725000.0,41.090909,0.0904,-0.5877,-0.1571,-0.65
53914,Garrett Acton,670183.0,actonga01,2023,OAK,1,AL,6,0,8,3.419,21,0.0049,NaN,35.137500,0.0549,-0.4476,0.0065,-0.39
53915,Jason Adam,592094.0,adamja01,2023,TBR,1,AL,56,0,21,27.641,124,0.0311,1775000.0,141.627778,0.5289,0.2856,-0.0905,0.72


In [4]:
bat_s = (bat[bat["PA"] > 0]
         .groupby(["mlb_ID", "year_ID"])
         .agg(name=("name_common", "first"), PA=("PA", "sum"), WAR=("WAR", "sum"))
         .reset_index())

pit_s = (pit.groupby(["mlb_ID", "year_ID"])
         .agg(name=("name_common", "first"), G=("G", "sum"), GS=("GS", "sum"),
              BIP=("BIP", "sum"), WAR=("WAR", "sum"))
         .reset_index())

print(bat_s[bat_s["year_ID"] >= 2022].groupby("year_ID")["PA"].sum())
print(bat_s[bat_s["year_ID"] == 2026].nlargest(5, "WAR")[["name", "PA", "WAR"]])

year_ID
2022    182052
2023    184104
2024    182449
2025    182926
2026    183849
Name: PA, dtype: int64
                      name   PA   WAR
89862  Pete Crow-Armstrong  726  9.83
88603         Brice Turang  680  7.34
90083      Kevin McGonigle  707  7.05
88922       Yordan Alvarez  692  6.77
89330       Bobby Witt Jr.  632  6.72


In [12]:
import pandas as pd
import unicodedata, re

def norm(s):
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode().lower()
    s = s.replace("-", " ")
    s = re.sub(r"\b(jr|sr|ii|iii|iv)\b\.?", "", s)
    s = re.sub(r"[^a-z ]", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    return {"michael fiers": "mike fiers", "steven pearce": "steve pearce"}.get(s, s)

bat_s["key"] = bat_s["name"].apply(norm)
pit_s["key"] = pit_s["name"].apply(norm)

c = pd.read_csv("../data/processed/contracts.csv")
c["key"] = c["player_name"].apply(norm)
c["stat_year"] = c["signing_year"] - 1

hit = c[c["player_type"] == "hitter"].merge(
    bat_s[["key", "year_ID", "mlb_ID", "WAR"]],
    left_on=["key", "stat_year"], right_on=["key", "year_ID"], how="left")
pitch = c[c["player_type"] == "pitcher"].merge(
    pit_s[["key", "year_ID", "mlb_ID", "WAR"]],
    left_on=["key", "stat_year"], right_on=["key", "year_ID"], how="left")

matched = pd.concat([hit, pitch], ignore_index=True)

print("Contracts:", len(c), "| Rows after join:", len(matched))

# Drop contracts where one name matched two different players
amb = matched.duplicated(["player_name", "signing_year"], keep=False)
print("Excluding", amb.sum(), "rows from",
      matched[amb][["player_name", "signing_year"]].drop_duplicates().shape[0],
      "ambiguous contracts")
matched = matched[~amb].copy()

print(matched.shape)
print(matched.groupby("player_type")["mlb_ID"].apply(lambda s: s.notna().mean()))

matched.to_csv("../data/processed/contracts_matched.csv", index=False)

Contracts: 1623 | Rows after join: 1628
Excluding 10 rows from 5 ambiguous contracts
(1618, 13)
player_type
hitter     0.950292
pitcher    0.887580
Name: mlb_ID, dtype: float64


In [13]:
seasons = {
    "hitter": bat_s[["mlb_ID", "year_ID", "key", "WAR"]],
    "pitcher": pit_s[["mlb_ID", "year_ID", "key", "WAR"]],
}

# 1. Fill in missing IDs by looking back up to 2 extra seasons
def find_id(row):
    if pd.notna(row["mlb_ID"]):
        return row["mlb_ID"]
    s = seasons[row["player_type"]]
    cand = s[(s["key"] == row["key"]) &
             (s["year_ID"].between(row["stat_year"] - 2, row["stat_year"]))]
    ids = cand["mlb_ID"].unique()
    return ids[0] if len(ids) == 1 else float("nan")

m = matched.copy()
m["mlb_ID"] = m.apply(find_id, axis=1)
print("Share with a player ID:", round(m["mlb_ID"].notna().mean(), 3))

# 2. Look up WAR for each of the three seasons before signing
lookup = {}
for t, s in seasons.items():
    for mid, yr, w in zip(s["mlb_ID"], s["year_ID"], s["WAR"]):
        lookup[(t, mid, yr)] = w

for back, col in enumerate(["war_1", "war_2", "war_3"]):
    m[col] = [
        lookup.get((t, i, y - back), 0.0) if pd.notna(i) else float("nan")
        for t, i, y in zip(m["player_type"], m["mlb_ID"], m["stat_year"])
    ]

# 3. Weighted average (most recent season counts most)
m["war_w"] = (5 * m["war_1"] + 4 * m["war_2"] + 3 * m["war_3"]) / 12

m.to_csv("../data/processed/contracts_with_war.csv", index=False)
print(m.shape)
print(m[["war_w", "war_1", "aav"]].corr()["aav"])
m.nlargest(8, "aav")[["player_name", "signing_year", "war_3", "war_2", "war_1", "war_w", "aav"]]

Share with a player ID: 0.959
(1618, 17)
war_w    0.743842
war_1    0.701687
aav      1.000000
Name: aav, dtype: float64


,player_name,signing_year,war_3,war_2,war_1,war_w,aav
1330,Shohei Ohtani,2024,4.07,6.25,3.80,4.684167,7.000000e+07
630,Kyle Tucker,2026,5.51,4.61,4.53,4.801667,6.000000e+07
561,Juan Soto,2025,5.47,5.29,8.00,6.464167,5.100000e+07
1163,Max Scherzer,2022,5.28,2.18,6.07,4.575833,4.333333e+07
1256,Justin Verlander,2023,0.16,0.00,5.61,2.377500,4.333333e+07
635,Bo Bichette,2026,4.75,-0.34,3.44,2.507500,4.200000e+07
434,Aaron Judge,2023,1.14,5.89,10.77,6.735833,4.000000e+07
563,Alex Bregman,2025,4.49,4.87,3.98,4.404167,4.000000e+07


In [14]:
from pybaseball import chadwick_register

reg = chadwick_register()
print(reg.shape)
print([c for c in reg.columns if "birth" in c.lower() or "mlbam" in c.lower()])
reg.head(3)

Gathering player lookup table. This may take a moment.
(26138, 8)
['key_mlbam']


,name_last,name_first,key_mlbam,key_retro,key_bbref,key_fangraphs,mlb_played_first,mlb_played_last
0,Bradley,Jed,605152,bradj002,bradlje01,13166,2016.0,2016.0
1,Barrios,Manuel,110625,barrm002,barrima01,1000605,1997.0,1998.0
2,Martin,Frank,118336,martf102,martifr01,1008165,1897.0,1899.0


In [15]:
import requests, time

ids = sorted(int(i) for i in m["mlb_ID"].dropna().unique())
print("Players to look up:", len(ids))

births = {}
for i in range(0, len(ids), 100):
    batch = ids[i:i+100]
    r = requests.get(
        "https://statsapi.mlb.com/api/v1/people",
        params={"personIds": ",".join(map(str, batch))},
        timeout=30,
    )
    r.raise_for_status()
    for p in r.json().get("people", []):
        if "birthDate" in p:
            births[p["id"]] = p["birthDate"]
    time.sleep(0.5)

print("Birth dates found:", len(births))

m["birth_date"] = pd.to_datetime(m["mlb_ID"].map(births))

# Age at signing: the offseason starts after season N, so use Dec 1 of the stat year
signing_date = pd.to_datetime(m["stat_year"].astype(int).astype(str) + "-12-01")
m["age_at_signing"] = ((signing_date - m["birth_date"]).dt.days / 365.25).round(1)

print(m["age_at_signing"].describe())
m.to_csv("../data/processed/contracts_with_age.csv", index=False)
m.nlargest(5, "aav")[["player_name", "signing_year", "birth_date", "age_at_signing", "aav"]]

Players to look up: 931
Birth dates found: 931
count    1552.000000
mean       32.191559
std         3.007202
min        23.700000
25%        30.100000
50%        31.800000
75%        34.000000
max        44.100000
Name: age_at_signing, dtype: float64


,player_name,signing_year,birth_date,age_at_signing,aav
1330,Shohei Ohtani,2024,1994-07-05,29.4,7.000000e+07
630,Kyle Tucker,2026,1997-01-17,28.9,6.000000e+07
561,Juan Soto,2025,1998-10-25,26.1,5.100000e+07
1163,Max Scherzer,2022,1984-07-27,37.3,4.333333e+07
1256,Justin Verlander,2023,1983-02-20,39.8,4.333333e+07


In [16]:
import numpy as np
from sklearn.linear_model import RidgeCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error

d = pd.read_csv("../data/processed/contracts_with_age.csv")
d = d[d["age_at_signing"].notna() & (d["player_name"] != "Shohei Ohtani")].copy()
d["aav_m"] = d["aav"] / 1e6
d["age_sq"] = (d["age_at_signing"] - 31) ** 2

def run_baseline(df, label):
    X = pd.get_dummies(
        df[["war_w", "age_at_signing", "age_sq", "qualifying_offer", "position"]],
        columns=["position"], dtype=float,
    )
    train = df["signing_year"] <= 2023
    test = df["signing_year"] >= 2024

    model = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-2, 3, 30)))
    model.fit(X[train], df.loc[train, "aav_m"])

    pred = model.predict(X[test])
    actual = df.loc[test, "aav_m"]
    naive = np.full(len(actual), df.loc[train, "aav_m"].median())

    print(f"{label} | train {train.sum()} | test {test.sum()}")
    print(f"  Model MAE: ${mean_absolute_error(actual, pred):.2f}M")
    print(f"  Naive MAE (always guess the median): ${mean_absolute_error(actual, naive):.2f}M")

run_baseline(d, "All contracts")
run_baseline(d[d["total_value"] >= 5_000_000], "$5M+ contracts only")

All contracts | train 1103 | test 448
  Model MAE: $3.53M
  Naive MAE (always guess the median): $5.62M
$5M+ contracts only | train 561 | test 213
  Model MAE: $4.38M
  Naive MAE (always guess the median): $6.57M


In [17]:
import numpy as np
import pandas as pd
from sklearn.linear_model import RidgeCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

d = pd.read_csv("../data/processed/contracts_with_age.csv")
d = d[d["age_at_signing"].notna() & (d["player_name"] != "Shohei Ohtani")].copy()
d["aav_m"] = d["aav"] / 1e6
d["age_sq"] = (d["age_at_signing"] - 31) ** 2

X = pd.get_dummies(
    d[["war_w", "age_at_signing", "age_sq", "qualifying_offer", "position"]],
    columns=["position"], dtype=float,
)
train = d["signing_year"] <= 2023
test = d["signing_year"] >= 2024

model = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-2, 3, 30)))
model.fit(X[train], d.loc[train, "aav_m"])

d["pred_m"] = model.predict(X)
d["error_m"] = d["pred_m"] - d["aav_m"]   # negative = model guessed too low

print("Average error by year (in $M, negative = guessed too low):")
print(d.groupby("signing_year")["error_m"].mean().round(2))

t = d[test].copy()
t["abs_err"] = t["error_m"].abs()
print("\nBiggest misses in the test years:")
print(t.nlargest(10, "abs_err")[
    ["player_name", "signing_year", "position", "war_w", "age_at_signing", "aav_m", "pred_m"]
].round(1))

Average error by year (in $M, negative = guessed too low):
signing_year
2015    0.59
2016    0.94
2017   -0.23
2018    0.67
2019    0.58
2020    0.84
2021    0.69
2022   -1.49
2023   -2.05
2024   -0.84
2025   -1.13
2026   -1.86
Name: error_m, dtype: float64

Biggest misses in the test years:
         player_name  signing_year position  war_w  age_at_signing  aav_m  \
626      Kyle Tucker          2026       RF    4.8            28.9   60.0   
631      Bo Bichette          2026       SS    2.5            27.7   42.0   
627     Alex Bregman          2026       3B    4.1            31.7   35.0   
1417     Blake Snell          2025       SP    3.6            32.0   36.4   
629      Pete Alonso          2026       1B    3.0            31.0   31.0   
557        Juan Soto          2025       RF    6.5            26.1   51.0   
1434  Walker Buehler          2025       SP   -0.4            30.3   21.0   
628   Cody Bellinger          2026       CF    4.0            30.4   32.5   
559     Alex B

In [18]:
from sklearn.metrics import mean_absolute_error

def fit_eval(df, use_year=False, use_log=False):
    cols = ["war_w", "age_at_signing", "age_sq", "qualifying_offer", "position"]
    X = pd.get_dummies(df[cols], columns=["position"], dtype=float)
    if use_year:
        X["year"] = df["signing_year"] - 2015
    tr = df["signing_year"] <= 2023
    te = df["signing_year"] >= 2024
    y = np.log(df["aav_m"]) if use_log else df["aav_m"]
    model = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-2, 3, 30)))
    model.fit(X[tr], y[tr])
    pred = model.predict(X[te])
    if use_log:
        pred = np.exp(pred)
    actual = df.loc[te, "aav_m"].values
    return mean_absolute_error(actual, pred), (pred - actual).mean()

variants = [
    ("Baseline", False, False),
    ("+ year trend", True, False),
    ("+ year trend, log target", True, True),
]

for label, subset in [("All contracts", d), ("$5M+ only", d[d["total_value"] >= 5_000_000])]:
    print(label)
    for name, uy, ul in variants:
        mae, bias = fit_eval(subset, uy, ul)
        print(f"  {name:28s} MAE ${mae:.2f}M | avg error ${bias:+.2f}M")

All contracts
  Baseline                     MAE $3.53M | avg error $-1.28M
  + year trend                 MAE $3.74M | avg error $+0.34M
  + year trend, log target     MAE $3.91M | avg error $-1.03M
$5M+ only
  Baseline                     MAE $4.38M | avg error $-2.65M
  + year trend                 MAE $4.00M | avg error $-0.16M
  + year trend, log target     MAE $4.05M | avg error $-1.19M


In [19]:
import numpy as np
import pandas as pd
from sklearn.linear_model import RidgeCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error

d = pd.read_csv("../data/processed/contracts_with_age.csv")
d = d[d["age_at_signing"].notna() & (d["player_name"] != "Shohei Ohtani")].copy()
d["aav_m"] = d["aav"] / 1e6
d["age_sq"] = (d["age_at_signing"] - 31) ** 2

# Season-by-season WAR lookup from the files saved earlier
bh = pd.read_csv("../data/processed/bwar_hitters.csv")
bp = pd.read_csv("../data/processed/bwar_pitchers.csv")
war_lookup = {}
for t, s in [("hitter", bh), ("pitcher", bp)]:
    for mid, yr, w in zip(s["mlb_ID"], s["year_ID"], s["WAR"]):
        war_lookup[(t, mid, yr)] = w

# Best single season in the 5 years before signing
d["peak5"] = [
    max(war_lookup.get((t, i, y - k), 0.0) for k in range(5))
    for t, i, y in zip(d["player_type"], d["mlb_ID"], d["stat_year"])
]

def fit_eval(df, extra=()):
    cols = ["war_w", "age_at_signing", "age_sq", "qualifying_offer", "position"] + list(extra)
    X = pd.get_dummies(df[cols], columns=["position"], dtype=float)
    X["year"] = df["signing_year"] - 2015
    tr = df["signing_year"] <= 2023
    te = df["signing_year"] >= 2024
    model = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-2, 3, 30)))
    model.fit(X[tr], df.loc[tr, "aav_m"])
    pred = model.predict(X[te])
    actual = df.loc[te, "aav_m"].values
    return mean_absolute_error(actual, pred), (pred - actual).mean()

for label, subset in [("All contracts", d), ("$5M+ only", d[d["total_value"] >= 5_000_000])]:
    print(label)
    for name, extra in [("Year trend", ()), ("Year trend + peak WAR", ("peak5",))]:
        mae, bias = fit_eval(subset, extra)
        print(f"  {name:24s} MAE ${mae:.2f}M | avg error ${bias:+.2f}M")

print(d[d["player_name"] == "Walker Buehler"][["signing_year", "war_w", "peak5", "aav_m"]])

All contracts
  Year trend               MAE $3.74M | avg error $+0.34M
  Year trend + peak WAR    MAE $3.71M | avg error $+0.19M
$5M+ only
  Year trend               MAE $4.00M | avg error $-0.16M
  Year trend + peak WAR    MAE $3.99M | avg error $-0.59M
      signing_year  war_w  peak5  aav_m
1434          2025  -0.39   7.13  21.05


In [20]:
from pybaseball import statcast_batter_percentile_ranks, statcast_pitcher_percentile_ranks

sav_b = statcast_batter_percentile_ranks(2023)
print(sav_b.shape)
print(sav_b.columns.tolist())

sav_p = statcast_pitcher_percentile_ranks(2023)
print(sav_p.shape)
print(sav_p.columns.tolist())

(614, 23)
['player_name', 'player_id', 'year', 'xwoba', 'xba', 'xslg', 'xiso', 'xobp', 'brl', 'brl_percent', 'exit_velocity', 'max_ev', 'hard_hit_percent', 'k_percent', 'bb_percent', 'whiff_percent', 'chase_percent', 'arm_strength', 'sprint_speed', 'oaa', 'bat_speed', 'squared_up_rate', 'swing_length']
(707, 22)
['player_name', 'player_id', 'year', 'xwoba', 'xba', 'xslg', 'xiso', 'xobp', 'brl', 'brl_percent', 'exit_velocity', 'max_ev', 'hard_hit_percent', 'k_percent', 'bb_percent', 'whiff_percent', 'chase_percent', 'arm_strength', 'xera', 'fb_velocity', 'fb_spin', 'curve_spin']


In [21]:
import pandas as pd, time
from pybaseball import statcast_batter_percentile_ranks, statcast_pitcher_percentile_ranks, cache

cache.enable()

hit_frames, pit_frames = [], []
for yr in range(2015, 2027):
    try:
        hb = statcast_batter_percentile_ranks(yr)
        hb["year"] = yr
        pp = statcast_pitcher_percentile_ranks(yr)
        pp["year"] = yr
        hit_frames.append(hb)
        pit_frames.append(pp)
        print(yr, "hitters:", len(hb), "pitchers:", len(pp))
    except Exception as e:
        print(yr, "FAILED:", str(e)[:100])
    time.sleep(1)

sav_hit = pd.concat(hit_frames, ignore_index=True)
sav_pit = pd.concat(pit_frames, ignore_index=True)
sav_hit.to_csv("../data/processed/savant_hitters.csv", index=False)
sav_pit.to_csv("../data/processed/savant_pitchers.csv", index=False)

# Are these 0-100 percentiles?
print(sav_hit[["xwoba", "exit_velocity"]].describe().loc[["min", "max"]])

# Share of players with a value for each metric, by year
cols = ["xwoba", "brl_percent", "exit_velocity", "chase_percent", "sprint_speed", "bat_speed"]
print(sav_hit.groupby("year")[cols].apply(lambda g: g.notna().mean()).round(2))

# Does player_id match the MLB IDs we already have?
d = pd.read_csv("../data/processed/contracts_with_age.csv")
ids = set(d["mlb_ID"].dropna().astype(int))
found = ids & (set(sav_hit["player_id"].astype(int)) | set(sav_pit["player_id"].astype(int)))
print("Contract players found in Savant data:", round(len(found) / len(ids), 3))

2015 hitters: 683 pitchers: 649
2016 hitters: 693 pitchers: 657
2017 hitters: 678 pitchers: 676
2018 hitters: 678 pitchers: 695
2019 hitters: 688 pitchers: 705
2020 hitters: 507 pitchers: 585
2021 hitters: 685 pitchers: 735
2022 hitters: 628 pitchers: 704
2023 hitters: 614 pitchers: 707
2024 hitters: 606 pitchers: 696
2025 hitters: 617 pitchers: 712
2026 hitters: 612 pitchers: 706
     xwoba  exit_velocity
min    1.0            1.0
max  100.0          100.0
      xwoba  brl_percent  exit_velocity  chase_percent  sprint_speed  \
year                                                                   
2015   0.37         0.37           0.37           0.37           1.0   
2016   0.35         0.35           0.35           0.35           1.0   
2017   0.38         0.38           0.38           0.38           1.0   
2018   0.37         0.37           0.37           0.37           1.0   
2019   0.36         0.36           0.36           0.36           1.0   
2020   0.50         0.50          

In [22]:
import numpy as np
import pandas as pd
from sklearn.linear_model import RidgeCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error

d = pd.read_csv("../data/processed/contracts_with_age.csv")
d = d[d["age_at_signing"].notna() & (d["player_name"] != "Shohei Ohtani")].copy()
d = d[d["total_value"] >= 5_000_000].copy()
d["aav_m"] = d["aav"] / 1e6
d["age_sq"] = (d["age_at_signing"] - 31) ** 2
d["year"] = d["signing_year"] - 2015

sav = {
    "hitter": pd.read_csv("../data/processed/savant_hitters.csv"),
    "pitcher": pd.read_csv("../data/processed/savant_pitchers.csv"),
}
feats = {
    "hitter": ["xwoba", "brl_percent", "exit_velocity", "chase_percent",
               "whiff_percent", "bb_percent", "sprint_speed"],
    "pitcher": ["xera", "fb_velocity", "k_percent", "bb_percent",
                "whiff_percent", "chase_percent", "hard_hit_percent"],
}

def savant_avg(df, t):
    s, f = sav[t], feats[t]
    lookup = dict(zip(zip(s["player_id"].astype(int), s["year"].astype(int)),
                      s[f].values.astype(float)))
    out = []
    for mid, sy in zip(df["mlb_ID"], df["stat_year"]):
        if pd.isna(mid):
            out.append([np.nan] * len(f))
            continue
        num, den = np.zeros(len(f)), np.zeros(len(f))
        for k, w in enumerate([5, 4, 3]):
            arr = lookup.get((int(mid), int(sy) - k))
            if arr is None:
                continue
            ok = ~np.isnan(arr)
            num[ok] += w * arr[ok]
            den[ok] += w
        out.append(np.where(den > 0, num / np.where(den > 0, den, 1), np.nan))
    return pd.DataFrame(out, columns=f, index=df.index)

def compare(t):
    df = d[d["player_type"] == t].copy()
    f = feats[t]
    total = len(df)
    df = pd.concat([df, savant_avg(df, t)], axis=1)
    df = df[df[f[0]].notna()].copy()      # require the main quality metric
    df[f] = df[f].fillna(50)              # other gaps: league-median percentile
    tr = (df["signing_year"] <= 2023).values
    te = ~tr
    print(f"\n{t.upper()}S: {len(df)} of {total} contracts have Savant data "
          f"| train {tr.sum()} | test {te.sum()}")

    def fit(cols):
        X = pd.get_dummies(df[cols + ["position"]], columns=["position"], dtype=float)
        model = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-2, 3, 30)))
        model.fit(X[tr], df.loc[tr, "aav_m"])
        return mean_absolute_error(df.loc[te, "aav_m"], model.predict(X[te]))

    base = ["age_at_signing", "age_sq", "qualifying_offer", "year"]
    naive = mean_absolute_error(df.loc[te, "aav_m"],
                                np.full(te.sum(), df.loc[tr, "aav_m"].median()))
    print(f"  Naive (median)              MAE ${naive:.2f}M")
    print(f"  A: WAR model                MAE ${fit(base + ['war_w']):.2f}M")
    print(f"  B: WAR + Savant             MAE ${fit(base + ['war_w'] + f):.2f}M")
    print(f"  C: Savant only (no WAR)     MAE ${fit(base + f):.2f}M")

compare("hitter")
compare("pitcher")


HITTERS: 291 of 328 contracts have Savant data | train 214 | test 77
  Naive (median)              MAE $7.94M
  A: WAR model                MAE $4.93M
  B: WAR + Savant             MAE $4.18M
  C: Savant only (no WAR)     MAE $4.94M

PITCHERS: 398 of 446 contracts have Savant data | train 271 | test 127
  Naive (median)              MAE $5.96M
  A: WAR model                MAE $3.69M
  B: WAR + Savant             MAE $3.43M
  C: Savant only (no WAR)     MAE $3.87M
